In [2]:
import yt
import unyt
from unyt import unyt_quantity, unyt_array, km, s, Mpc, kg, m, Msun, kpc
from unyt import Gyr, Mpc, km, s
from unyt.physical_constants import G as G_unyt, kboltz, mp as mp_unyt  # G, k_B, m_p

import numpy as np
import matplotlib.pyplot as plt
import caesar
import pynbody as pnb
import pynbody.units as u
import pynbody.array as pna

import dill
import os
from astro_constants import NA_no_units, NA, kB_no_units, kB, mH, mp_no_units, mp, mu_e, mu, G_no_units, G
#from astro_constants import NA_no_units, NA, kB_no_units, kB, mH, mp_no_units, mp, mu_e, mu, G_no_units,

code = 'Simba'

target_halo = 3224
snap_file = '/scratch/b/babul/aspadawe/snapshots/Hyenasc_L1_zooms/halo_3224/snap_halo_3224_151.hdf5'

# ADD CAESAR FILEPATH HERE TOO

snap = yt.load(snap_file)#, unit_system='cgs')
s = pnb.load(snap_file)
gas_positions = s.gas['pos']

id_filepath = "/home/b/babul/stlock/entropy_profiles/finished/ids_formatted.txt" # containing halo data/ids

# obj = caesar.load('/scratch/b/babul/aspadawe/snapshots/Hyenasc_L1_zooms/halo_2919/Groups/Caesar_halo_2919_151.hdf5')
obj = caesar.load('/scratch/b/babul/aspadawe/snapshots/Hyenasc_L1_zooms/halo_3224/Groups/Caesar_halo_3224_151.hdf5')
halo_data = obj.halos


yt : [INFO     ] 2025-03-26 03:30:47,937 ComovingIntegrationOn does not exist, falling back to OmegaLambda
yt : [INFO     ] 2025-03-26 03:30:47,951 Calculating time from 1.000e+00 to be 4.375e+17 seconds
yt : [INFO     ] 2025-03-26 03:30:47,952 Assuming length units are in kpc/h (comoving)
yt : [INFO     ] 2025-03-26 03:30:48,038 Parameters: current_time              = 4.3748565942961094e+17 s
yt : [INFO     ] 2025-03-26 03:30:48,038 Parameters: domain_dimensions         = [1 1 1]
yt : [INFO     ] 2025-03-26 03:30:48,039 Parameters: domain_left_edge          = [0. 0. 0.]
yt : [INFO     ] 2025-03-26 03:30:48,039 Parameters: domain_right_edge         = [200000. 200000. 200000.]
yt : [INFO     ] 2025-03-26 03:30:48,040 Parameters: cosmological_simulation   = True
yt : [INFO     ] 2025-03-26 03:30:48,040 Parameters: current_redshift          = 4.440892098500626e-16
yt : [INFO     ] 2025-03-26 03:30:48,041 Parameters: omega_lambda              = 0.7
yt : [INFO     ] 2025-03-26 03:30:48,041 

In [3]:
def get_coordinates_by_id(id_filepath, target_id):
    with open(id_filepath, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 6:
                continue  # skip malformed lines
            
            try:
                obj_id = int(parts[2])
                if obj_id == target_id:
                    coord_str = parts[4].strip('()') # parts[4] - contains the coordinates
                    x, y, z = map(float, coord_str.split(','))
                    return (x, y, z)
            except (ValueError, IndexError):
                continue  # skip malformed lines
    
    return None  # if ID not found

In [4]:
coords = get_coordinates_by_id(id_filepath, target_halo)
halo_center = unyt_array(coords, 'kpc')
#halo_center/=1000
print(halo_center)

[103161.32  136991.281 104131.414] kpc


In [5]:
min = halo_data[0]
for halo in halo_data:
    if halo.minpotpos[0] > min.minpotpos[0]:
        min = halo
print(min.minpotpos)

[164483.66 137774.22 137260.12] kpccm


In [52]:
target_coords = np.array([103161.32,136991.281, 104131.414])  # assumed physical kpc
min_dist = float('inf')
closest_halo = None

for i, halo in enumerate(obj.halos):
    halo_pos = np.array(halo.minpotpos)
    dist = np.linalg.norm(halo_pos - target_coords)
    if dist < min_dist:
        min_dist = dist
        closest_halo = halo

print(f"Closest halo is at distance: {min_dist:.2f} kpc")
print("Closest halo position:", closest_halo.minpotpos)
print("Your target position:", target_coords)


Closest halo is at distance: 47151.04 kpc
Closest halo position: [138116.28 144346.34 134909.23] kpccm
Your target position: [103161.32  136991.281 104131.414]


In [5]:
print(len(halo_data))

6617
